# Solace Browser Deployment Notebook 02

NORTHSTAR: promote verified GitHub native artifacts to GCS for Linux/macOS/Windows with checksum integrity (Windows artifact must be signed MSI).


## Inputs

- Successful GitHub Actions run from `build-binaries`.
- GitHub credentials available via `git credential fill`.
- `gcloud` already authenticated and active project configured.
- Write access to `gs://solace-downloads`.


In [ ]:
import hashlib
import os
import shutil
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'VERSION').is_file() and (candidate / 'src/scripts/promote_native_builds_to_gcs.py').is_file():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')


def resolve_cli(name: str) -> str:
    candidates = [name]
    if os.name == 'nt':
        candidates = [f'{name}.cmd', f'{name}.exe', name]
    for candidate in candidates:
        resolved = shutil.which(candidate)
        if resolved:
            return resolved
    if os.name == 'nt':
        cloud_sdk = Path.home() / 'AppData' / 'Local' / 'Google' / 'Cloud SDK' / 'google-cloud-sdk' / 'bin' / f'{name}.cmd'
        if cloud_sdk.is_file():
            return str(cloud_sdk)
    raise RuntimeError(f'{name} CLI not found on PATH')


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT_PATH = REPO_ROOT / 'src/scripts/promote_native_builds_to_gcs.py'
GCLOUD = resolve_cli('gcloud')
PYTHON_BIN = sys.executable

print(f'REPO_ROOT={REPO_ROOT}')
print(f'PYTHON_BIN={PYTHON_BIN}')
print(f'GCLOUD={GCLOUD}')
print(subprocess.run([PYTHON_BIN, '--version'], capture_output=True, text=True, check=True).stdout.strip())
print(subprocess.run([GCLOUD, '--version'], capture_output=True, text=True, check=True).stdout.splitlines()[0])
assert SCRIPT_PATH.is_file(), f'Missing promotion script: {SCRIPT_PATH}'

active_account = subprocess.run(
    [GCLOUD, 'auth', 'list', '--filter=status:ACTIVE', "--format=value(account)"],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
project_id = subprocess.run(
    [GCLOUD, 'config', 'get-value', 'project'],
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()

assert active_account, 'No active gcloud account'
assert project_id and project_id != '(unset)', 'gcloud project is not configured'
subprocess.run([GCLOUD, 'storage', 'ls', 'gs://solace-downloads'], check=True, stdout=subprocess.DEVNULL)
print(f'OK: gcloud active account={active_account} project={project_id}')


In [ ]:
import os
import subprocess
from pathlib import Path

version = (REPO_ROOT / 'VERSION').read_text(encoding='utf-8').strip()
version_tag = f'v{version}'
run_id_file = REPO_ROOT / 'notebooks' / 'deployment' / '.last_build_run_id'

cmd = [PYTHON_BIN, str(SCRIPT_PATH)]
if run_id_file.is_file():
    run_id = run_id_file.read_text(encoding='utf-8').strip()
    print(f'Promoting artifacts from explicit run id {run_id}')
    cmd.extend(['--run-id', run_id, '--tag', version_tag])
else:
    print(f'Promoting artifacts for tag {version_tag}')
    cmd.extend(['--tag', version_tag])

env = os.environ.copy()
env['PATH'] = f"{Path(GCLOUD).parent}{os.pathsep}" + env.get('PATH', '')
subprocess.run(cmd, cwd=REPO_ROOT, env=env, check=True)


In [ ]:
import hashlib
import struct
import tempfile
import urllib.request
from pathlib import Path

base = 'https://storage.googleapis.com/solace-downloads/solace-browser/latest'
urls = {
    'linux': f'{base}/solace-browser-linux-x86_64',
    'macos': f'{base}/solace-browser-macos-universal',
    'windows': f'{base}/solace-browser-windows-x86_64.msi',
    'linux_sha': f'{base}/solace-browser-linux-x86_64.sha256',
    'macos_sha': f'{base}/solace-browser-macos-universal.sha256',
    'windows_sha': f'{base}/solace-browser-windows-x86_64.msi.sha256',
}

for label, url in urls.items():
    with urllib.request.urlopen(url, timeout=60) as resp:
        code = getattr(resp, 'status', 200)
    print(f'{code} {url}')
    assert code == 200, f'{label} URL returned HTTP {code}'


def download(url: str, path: Path) -> None:
    with urllib.request.urlopen(url, timeout=120) as resp:
        path.write_bytes(resp.read())


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def parse_sha_file(path: Path) -> tuple[str, str]:
    parts = path.read_text(encoding='utf-8').strip().split()
    if len(parts) < 2:
        raise RuntimeError(f'invalid sha256 file format: {path}')
    return parts[0], parts[-1]

with tempfile.TemporaryDirectory(prefix='solace-promotion-verify-') as tmp_dir:
    tmp = Path(tmp_dir)
    linux = tmp / 'linux'
    mac = tmp / 'macos'
    win = tmp / 'windows.msi'
    linux_sha_file = tmp / 'linux.sha256'
    mac_sha_file = tmp / 'macos.sha256'
    win_sha_file = tmp / 'windows.sha256'

    download(urls['linux'], linux)
    download(urls['macos'], mac)
    download(urls['windows'], win)
    download(urls['linux_sha'], linux_sha_file)
    download(urls['macos_sha'], mac_sha_file)
    download(urls['windows_sha'], win_sha_file)

    assert linux.read_bytes().startswith(b'\x7fELF'), 'linux latest is not ELF'
    mach_o_headers = {
        b'\xfe\xed\xfa\xce',
        b'\xce\xfa\xed\xfe',
        b'\xfe\xed\xfa\xcf',
        b'\xcf\xfa\xed\xfe',
        b'\xca\xfe\xba\xbe',
        b'\xbe\xba\xfe\xca',
        b'\xca\xfe\xba\xbf',
        b'\xbf\xba\xfe\xca',
    }
    assert mac.read_bytes()[:4] in mach_o_headers, 'mac latest is not Mach-O'

    windows_bytes = win.read_bytes()
    ole2_header = b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
    assert windows_bytes.startswith(ole2_header), 'windows latest is not MSI (OLE2 header missing)'

    linux_sha, _ = parse_sha_file(linux_sha_file)
    mac_sha, _ = parse_sha_file(mac_sha_file)
    win_sha, win_name = parse_sha_file(win_sha_file)

    checks = {
        'linux_sha_match': sha256(linux) == linux_sha,
        'mac_sha_match': sha256(mac) == mac_sha,
        'windows_sha_match': sha256(win) == win_sha,
        'windows_sha_name_ok': win_name == 'solace-browser-windows-x86_64.msi',
    }
    assert all(checks.values()), f'sha mismatch: {checks}'
    print({
        'status': 'ok',
        **checks,
        'windows_installer_size_bytes': len(windows_bytes),
        'windows_installer_size_mb': round(len(windows_bytes) / (1024 * 1024), 2),
    })


## Expected output artifacts

- `scratch/native-promotion/<timestamp>/promotion-summary.json`
- `scratch/native-promotion/<timestamp>/` extracted artifact zips
- GCS objects at both `v{VERSION}` and `latest` keys
- Public `latest` download URLs returning Linux ELF, macOS Mach-O, and Windows installer PE
- If Windows artifact is not installer, promotion fails before any upload (atomic fail-closed gate)
